In [ ]:
import numpy as np
import math
import rclpy
import threading
import time

from semantic_digital_twin.world import World
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
from semantic_digital_twin.robots.soft_trunk import SoftTrunk


In [ ]:
# PCC TEST
# Initialize ROS 2 Node
if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node("soft_pcc_test")
thread = threading.Thread(target=rclpy.spin, args=(node,), daemon=True)
thread.start()

In [ ]:
# Create the snake with 3 sections, and 10 visual segments per section
num_sections = 5
world = World()
trunk, kappas, phis = SoftTrunk.build_pcc(world, num_sections=num_sections, segs_per_section=10, total_robot_length=2.0)

# Setup Visualization
tf_pub = TFPublisher(_world=world, node=node)
viz_pub = VizMarkerPublisher(_world=world, node=node)
tf_pub.add_to_world(world)
viz_pub.add_to_world(world)

print(f"Robot '{trunk.name.name}' is ready. Check RViz.")


In [ ]:
# set the fixed frame to snake/base in rviz
try:
    while True:
        print("Animating Snake: Propagating Wave...")
        for t in np.linspace(0, 20, 400):
            
            for s in range(num_sections):
                # The 's * 1.0' creates a delay so section 2 starts bending 
                # slightly after section 1.
                k_val = 1.5 * np.sin(t - s * 1.0) 
                
                # make it spiral by rotating the phis
                p_val = t * 0.5 
                
                # Update the specific DOFs for this section
                world.state[kappas[s].id].position = float(k_val)
                world.state[phis[s].id].position = float(p_val)
            
            # Recompute everything
            world.notify_state_change()
            
            # Update RViz
            tf_pub.notify()
            viz_pub.notify()
            
            time.sleep(0.03)
        
except KeyboardInterrupt:
    print("Animation stopped.")

In [ ]:
node.destroy_node()
rclpy.shutdown()

In [ ]:
# COSSERAT ROD TEST
# Initialize ROS 2 Node
if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node("cosserat_test")
thread = threading.Thread(target=rclpy.spin, args=(node,), daemon=True)
thread.start()

In [ ]:
world = World()
# Build the Cosserat version
trunk, ux_dof, uy_dof, uz_dof = SoftTrunk.build_cosserat_trunk(world)

tf_pub = TFPublisher(_world=world, node=node)
viz_pub = VizMarkerPublisher(_world=world, node=node)
tf_pub.add_to_world(world)
viz_pub.add_to_world(world)

print(f"Cosserat Robot '{trunk.name.name}' is ready.")

In [ ]:
# set the fixed frame to cosserat/base in rviz
try:
    print("Animating Cosserat: Bending and Twisting...")
    while True:
        for t in np.linspace(0, 10, 200):
            # Bending: Create a circular motion using ux and uy
            # Values between -2 and 2
            val_ux = 2.0 * np.sin(t)
            val_uy = 2.0 * np.cos(t)
            
            # Torsion: Twist the rod back and forth
            val_uz = 1.5 * np.sin(t * 0.5)
            
            # Update World State
            world.state[ux_dof.id].position = float(val_ux)
            world.state[uy_dof.id].position = float(val_uy)
            world.state[uz_dof.id].position = float(val_uz)
            
            # Notify and Publish
            world.notify_state_change()
            tf_pub.notify()
            viz_pub.notify()
            
            time.sleep(0.05)
except KeyboardInterrupt:
    print("Stopped.")

In [ ]:
node.destroy_node()
rclpy.shutdown()